# AION-C · MoSE Pipeline Benchmark

**Mixture of Specialized Engines — end-to-end routing verification.**

Verifica que el pipeline completo funciona con pesos random:
- Orchestrator selecciona el motor correcto (vía heurísticas de keywords)
- Cada motor cristaliza un grafo semántico del dominio
- Unifier fusiona representaciones de motores activos
- Decoder produce logits sobre el vocabulario

**No hay entrenamiento.** Solo forward pass con pesos aleatorios.

| Motor   | Dominio     | Keywords heurísticas (ejemplos)       |
|---------|-------------|---------------------------------------|
| CORA    | Causal      | default (ninguna → CORA)              |
| FORGE-C | Código      | código, function, class, python, def… |
| AXIOM   | Matemáticas | demuestra, teorema, calcula, proof…   |
| MUSE    | Narrativa   | historia, personaje, héroe, cuento…   |
| EMPATHY | Social      | cree, siente, empatía, emoción…       |

In [ ]:
# ── Configura la ruta al repo ─────────────────────────────────────────────────
#
# Opción A: Google Drive
#   from google.colab import drive
#   drive.mount('/content/drive')
#   REPO_PATH = '/content/drive/MyDrive/AION-C'
#
# Opción B: git clone
#   !git clone https://github.com/TU_USUARIO/AION-C.git /content/AION-C
#   REPO_PATH = '/content/AION-C'

REPO_PATH = '/content/AION-C'   # ← cambia según tu setup

import sys, os, time
sys.path.insert(0, REPO_PATH)
os.chdir(REPO_PATH)

import torch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Python {sys.version.split()[0]}')
print(f'PyTorch {torch.__version__}')
print(f'Device  {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU     {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Build pipeline ────────────────────────────────────────────────────────────

from router.pipeline import MoSEPipeline, MoSEConfig

torch.manual_seed(42)
cfg      = MoSEConfig.tiny()   # hidden_dim=64, vocab_size=512
pipeline = MoSEPipeline(cfg).to(DEVICE).eval()

bd = pipeline.parameter_breakdown()
W  = 38
print(f"{'─'*W}")
print(f"  {'Module':<22} {'Params':>10}")
print(f"{'─'*W}")
for k, v in bd.items():
    if k != 'total_unique':
        print(f"  {k:<22} {v:>10,}")
print(f"{'─'*W}")
print(f"  {'TOTAL (unique)':<22} {bd['total_unique']:>10,}")
print(f"{'─'*W}")

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def tokenize(text, vocab_size=512, max_len=48, device='cpu'):
    """Tokenizador de caracteres mínimo: ord(c) % vocab_size."""
    ids = [ord(c) % vocab_size for c in text[:max_len]]
    ids += [0] * (max_len - len(ids))
    return torch.tensor([ids], dtype=torch.long, device=device)


def fmt_graph(graph):
    """Formatea nodos y aristas de un CausalGraph para display."""
    lines = []

    nodes = graph.nodes
    edges = graph.edges

    if not nodes:
        lines.append('    (no nodes)')
        return '\n'.join(lines)

    lines.append(f'    Nodes ({len(nodes)}):')
    for nd in nodes[:10]:
        ntype = nd.node_type.value if hasattr(nd.node_type, 'value') else str(nd.node_type)
        lines.append(f'      {nd.node_id:<6}  [{ntype:<16}]  conf={nd.confidence:.2f}')
    if len(nodes) > 10:
        lines.append(f'      ... +{len(nodes)-10} more')

    if edges:
        lines.append(f'    Edges ({len(edges)}):')
        for ed in edges[:10]:
            rel = ed.relation.value if hasattr(ed.relation, 'value') else str(ed.relation)
            lines.append(f'      {ed.source_id} ──[{rel}]──▶ {ed.target_id}')
        if len(edges) > 10:
            lines.append(f'      ... +{len(edges)-10} more')
    else:
        lines.append('    Edges (0): (threshold not exceeded with random weights)')

    return '\n'.join(lines)


def run_query(pipeline, query, cfg, label):
    """Forward pass completo + inspección de routing y grafo."""
    t0  = time.perf_counter()
    ids = tokenize(query, vocab_size=cfg.vocab_size, max_len=48, device=DEVICE)

    with torch.no_grad():
        # Paso a paso para poder inspeccionar el grafo
        concepts  = pipeline.encoder(ids)                              # [1, L, D]
        orch_out  = pipeline.orchestrator(concepts, query_text=query)  # routing
        primary   = orch_out.primary_motor
        motor     = pipeline.motors[primary.motor_name]
        cryst_out = motor.build_graph(concepts)                        # crystallizer
        graph     = cryst_out.graphs[0]                               # primer item

        # Forward completo para logits y confianza
        out = pipeline(ids, query_text=query)

    elapsed_ms = (time.perf_counter() - t0) * 1000

    # Top tokens del último position (predicción del siguiente token)
    top5_ids = out.logits[0, -1].topk(5).indices.tolist()

    # Motores secundarios (si los hay)
    secondary = [
        f"{a.motor_name}({a.score:.2f}×{a.n_iterations}it)"
        for a in orch_out.activations[1:]
    ]

    W = 66
    print(f"\n{'━'*W}")
    print(f"  {label}")
    print(f"  Q: {query[:62]}{'…' if len(query)>62 else ''}")
    print(f"{'─'*W}")
    print(f"  Routing  │ mode={orch_out.routing_mode}")
    print(f"  Motor    │ {primary.motor_name.upper():<10}  "
          f"score={primary.score:.3f}  iters={primary.n_iterations}")
    if secondary:
        print(f"  Also     │ {', '.join(secondary)}")
    print(f"{'─'*W}")
    print(f"  Graph [{primary.motor_name}]  "
          f"nodes={len(graph.nodes)}  edges={len(graph.edges)}")
    print(fmt_graph(graph))
    print(f"{'─'*W}")
    print(f"  Decoder  │ top-5 token ids = {top5_ids}")
    print(f"           │ confidence      = {out.confidence[0].item():.4f}")
    print(f"           │ needs_clarif    = {bool(out.needs_clarif[0].item())}")
    print(f"  Elapsed  │ {elapsed_ms:.1f} ms")

In [ ]:
# ── 5 Queries — una por dominio ───────────────────────────────────────────────

QUERIES = [
    (
        "Query 1 · CAUSAL  (esperado: CORA)",
        "Si la lluvia causa suelo mojado y suelo mojado causa inundación, "
        "¿la lluvia causa inundación?",
    ),
    (
        "Query 2 · CÓDIGO  (esperado: FORGE-C)",
        "La función getData llama a parseJSON que llama a validate. "
        "¿Qué código se afecta si cambio validate?",
    ),
    (
        "Query 3 · MATH    (esperado: AXIOM)",
        "Si a mayor que b y b mayor que c, ¿a mayor que c? "
        "Demuestra usando el teorema de transitividad.",
    ),
    (
        "Query 4 · NARRATIVA (esperado: MUSE)",
        "El héroe desea libertad pero el villano lo impide con un conflicto. "
        "¿Cómo se resuelve la historia del personaje?",
    ),
    (
        "Query 5 · SOCIAL  (esperado: EMPATHY)",
        "Ana cree que Pedro está enojado pero Pedro realmente está triste. "
        "¿Hay un malentendido emocional entre ellos?",
    ),
]

t_start = time.perf_counter()

for label, query in QUERIES:
    run_query(pipeline, query, cfg, label=label)

total_ms = (time.perf_counter() - t_start) * 1000
print(f"\n{'━'*66}")
print(f"  TOTAL  {total_ms:.0f} ms  │  {len(QUERIES)} queries  │  device: {DEVICE}")
print(f"{'━'*66}")

In [ ]:
# ── Tabla resumen de routing ──────────────────────────────────────────────────

print(f"\n  {'Query':<12} {'Esperado':<10} {'Seleccionado':<14} {'Score':>7}  {'Mode'}")
print(f"  {'─'*58}")

expected = ['cora', 'forge_c', 'axiom', 'muse', 'empathy']

with torch.no_grad():
    for i, (label, query) in enumerate(QUERIES):
        ids      = tokenize(query, vocab_size=cfg.vocab_size, device=DEVICE)
        concepts = pipeline.encoder(ids)
        orch     = pipeline.orchestrator(concepts, query_text=query)
        pri      = orch.primary_motor
        match    = '✓' if pri.motor_name == expected[i] else '✗'
        qname    = label.split('·')[1].strip().split()[0]
        print(f"  {qname:<12} {expected[i]:<10} {pri.motor_name:<14} "
              f"{pri.score:>7.3f}  {orch.routing_mode}  {match}")

In [ ]:
# ── Gradient check: pipeline es entrenable ────────────────────────────────────

pipeline.train()

ids = tokenize(QUERIES[0][1], vocab_size=cfg.vocab_size, device=DEVICE)
out = pipeline(ids, query_text=QUERIES[0][1])

loss = out.logits.sum()
loss.backward()

enc_grad = pipeline.encoder.token_embedding.weight.grad
print(f"Backward OK")
print(f"  loss value         = {loss.item():.4f}")
print(f"  embedding grad     = {'✓' if enc_grad is not None else '✗'}")
print(f"  embedding grad norm= {enc_grad.norm().item():.6f}")

# Verifica gradientes en todos los motores
for name, motor in pipeline.motors.items():
    has_grad = any(p.grad is not None for p in motor.parameters() if p.requires_grad)
    print(f"  motor[{name:<8}]  grad = {'✓' if has_grad else '○ (not in active path)'  }")

pipeline.eval()
print('\nDone.')